# Task 2.2 — 불만 유형 분류 분석

## 분석 방향
리뷰에서 불만을 **3가지 유형**으로 분리:
1. **제품 불만** — 성능/기능/디자인/가성비 등 제품 자체
2. **쿠팡 불만** — 배송/포장/배송업체/고객서비스
3. **브랜드/기종 불만** — 삼성 vs 애플 비교, 특정 모델 이슈

Rating은 참고하되 100% 신뢰하지 않음 → 텍스트 기반으로 불만 유형 직접 판별

In [ ]:
import re, json
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

DATA_PATH = Path('../si_dataset/review_for_analysis.json')

In [ ]:
# 데이터 로드
with open(DATA_PATH, encoding='utf-8') as f:
    raw = f.read()
raw = re.sub(r':\s*NaN\b', ': null', raw)
raw = re.sub(r'\bnull([A-Z][a-zA-Z]*)', 'null', raw)
records = json.loads(raw)

df = pd.DataFrame(records)
df['brand'] = df['product_name'].apply(lambda x: 'Apple' if 'iphone' in x else 'Samsung')
df['content_len'] = df['content'].fillna('').str.len()
df['text'] = (df['title'].fillna('') + ' ' + df['content'].fillna('')).str.strip()

print(f'총 리뷰: {len(df):,}개')
print(df['product_name'].value_counts().to_string())

## 1. 키워드 기반 불만 유형 분류기

Rating에 의존하지 않고, **텍스트에서 직접** 불만 유형을 판별합니다.

In [ ]:
# 불만 유형별 키워드 사전
COMPLAINT_KEYWORDS = {
    'coupang': [
        '배송', '택배', '포장', '박스', '배달', '배송기사', '쿠팡맨', '로켓배송',
        '배송지연', '늦게', '언제와', '파손', '찌그러', '찍힘', '긁힘',
        '박스가', '박스는', '밀봉', '테이프', '뜯겨', '구겨', '훼손',
        '배송완료', '문앞', '반품', '교환', '환불', '고객센터', '쿠팡'
    ],
    'product': [
        '성능', '배터리', '카메라', '발열', '속도', '렉', '끊김', '느림',
        '화면', '디스플레이', '액정', '스크린', '화질',
        '무게', '두께', '크기', '디자인', '그립감', '촉감',
        '가성비', '가격', '비싸', '가격대비',
        '기능', '충전', '음질', '스피커', '진동', '지문인식', '페이스아이디',
        '업데이트', '버그', '오류', '에러', '고장', '불량', '결함',
        'AS', 'A/S', '수리', '서비스센터'
    ],
    'brand_model': [
        '삼성', '갤럭시', '아이폰', '애플', 'iphone', 'samsung', 'galaxy',
        '안드로이드', 'ios', '아이클라우드', '갤럭시AI', '원UI',
        '전작', '전모델', '이전작', '비교', '보다 나은', '보다 못한',
        's25', 's24', '아이폰16', '아이폰15', 'pro max', 'ultra',
        '폴드', '플립', 'fold', 'flip'
    ]
}

def classify_complaint(text):
    """텍스트에서 불만 유형 분류 (중복 허용)"""
    text_lower = text.lower()
    found = []
    for category, keywords in COMPLAINT_KEYWORDS.items():
        if any(kw in text_lower for kw in keywords):
            found.append(category)
    return found if found else ['unknown']

df['complaint_types'] = df['text'].apply(classify_complaint)

# 각 유형별 플래그 컬럼
for cat in ['coupang', 'product', 'brand_model']:
    df[f'is_{cat}'] = df['complaint_types'].apply(lambda x: cat in x)

print('불만 유형 분포:')
for cat in ['coupang', 'product', 'brand_model']:
    n = df[f'is_{cat}'].sum()
    print(f'  {cat}: {n:,}개 ({n/len(df)*100:.1f}%)')
print(f'  unknown (키워드 없음): {(df["complaint_types"].apply(lambda x: x == ["unknown"])).sum():,}개')

## 2. Rating과 불만 유형의 관계

Rating이 낮다고 해서 반드시 제품 불만은 아님 → 쿠팡 배송 불만이 낮은 별점을 만드는 경우를 분리

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, cat in zip(axes, ['coupang', 'product', 'brand_model']):
    # 해당 유형 언급 vs 미언급 별점 분포
    yes = df[df[f'is_{cat}']]['rating'].value_counts().sort_index()
    no  = df[~df[f'is_{cat}']]['rating'].value_counts().sort_index()
    
    # 비율로 정규화
    yes_pct = yes / yes.sum() * 100
    no_pct  = no  / no.sum()  * 100
    
    x = np.arange(1, 6)
    w = 0.35
    ax.bar(x - w/2, [yes_pct.get(i, 0) for i in x], w, label='언급', color='#e53935', alpha=0.8)
    ax.bar(x + w/2, [no_pct.get(i, 0)  for i in x], w, label='미언급', color='#1e88e5', alpha=0.8)
    
    label_map = {'coupang': '쿠팡/배송', 'product': '제품', 'brand_model': '브랜드/기종'}
    ax.set_title(f'{label_map[cat]} 불만 언급 여부별 별점 분포')
    ax.set_xlabel('별점')
    ax.set_ylabel('%')
    ax.legend()
    
    mean_yes = df[df[f'is_{cat}']]['rating'].mean()
    mean_no  = df[~df[f'is_{cat}']]['rating'].mean()
    ax.set_xlabel(f'별점  (언급 평균: {mean_yes:.2f} / 미언급: {mean_no:.2f})')

plt.tight_layout()
plt.show()

In [ ]:
# 낮은 별점(1-2점) 리뷰 중 불만 유형 분포
low_rating = df[df['rating'] <= 2]
high_rating = df[df['rating'] >= 4]

print(f'저점(1-2점) 리뷰: {len(low_rating):,}개')
print(f'고점(4-5점) 리뷰: {len(high_rating):,}개')
print()

cats = ['coupang', 'product', 'brand_model']
labels = ['쿠팡/배송', '제품', '브랜드/기종']

low_pcts  = [low_rating[f'is_{c}'].mean()*100 for c in cats]
high_pcts = [high_rating[f'is_{c}'].mean()*100 for c in cats]

x = np.arange(len(cats))
fig, ax = plt.subplots(figsize=(8, 4))
w = 0.35
ax.bar(x - w/2, low_pcts,  w, label='저점(1-2점)', color='#e53935', alpha=0.85)
ax.bar(x + w/2, high_pcts, w, label='고점(4-5점)', color='#43a047', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('언급 비율 (%)')
ax.set_title('별점 구간별 불만 유형 언급 비율')
ax.legend()
for i, (lv, hv) in enumerate(zip(low_pcts, high_pcts)):
    ax.text(i - w/2, lv + 0.5, f'{lv:.1f}%', ha='center', fontsize=9)
    ax.text(i + w/2, hv + 0.5, f'{hv:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. 쿠팡 불만 vs 제품 불만 — 별점 왜곡 분석

"쿠팡 불만만 있는" 저점 리뷰 = 제품과 무관하게 별점이 낮아진 케이스

In [ ]:
# 불만 유형 조합별 분류
def complaint_label(row):
    c = row['is_coupang']
    p = row['is_product']
    b = row['is_brand_model']
    if c and not p and not b: return '쿠팡만'
    if p and not c and not b: return '제품만'
    if b and not c and not p: return '브랜드만'
    if c and p: return '쿠팡+제품'
    if c and b: return '쿠팡+브랜드'
    if p and b: return '제품+브랜드'
    if c and p and b: return '전체'
    return '없음'

df['complaint_label'] = df.apply(complaint_label, axis=1)

label_order = ['쿠팡만', '제품만', '브랜드만', '쿠팡+제품', '쿠팡+브랜드', '제품+브랜드', '전체', '없음']
group_stats = df.groupby('complaint_label').agg(
    리뷰수=('rating', 'count'),
    평균별점=('rating', 'mean'),
    별점1_2비율=('rating', lambda x: (x <= 2).mean() * 100)
).reindex(label_order).dropna()

print(group_stats.round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

colors = ['#e53935', '#1e88e5', '#43a047', '#fb8c00', '#8e24aa', '#00acc1', '#6d4c41', '#757575']

# 평균 별점
group_stats['평균별점'].plot(kind='bar', ax=axes[0], color=colors[:len(group_stats)], alpha=0.85)
axes[0].set_title('불만 유형 조합별 평균 별점')
axes[0].set_ylabel('평균 별점')
axes[0].set_ylim(1, 5)
axes[0].tick_params(axis='x', rotation=30)
axes[0].axhline(df['rating'].mean(), color='gray', linestyle='--', alpha=0.6, label=f'전체 평균 {df["rating"].mean():.2f}')
axes[0].legend()

# 저점 비율
group_stats['별점1_2비율'].plot(kind='bar', ax=axes[1], color=colors[:len(group_stats)], alpha=0.85)
axes[1].set_title('불만 유형 조합별 1-2점 비율 (%)')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# 인사이트 출력
if '쿠팡만' in group_stats.index and '제품만' in group_stats.index:
    coupang_only_low = group_stats.loc['쿠팡만', '별점1_2비율']
    product_only_low = group_stats.loc['제품만', '별점1_2비율']
    print(f'\n[인사이트] 쿠팡 불만만 있는 리뷰의 저점 비율: {coupang_only_low:.1f}%')
    print(f'[인사이트] 제품 불만만 있는 리뷰의 저점 비율: {product_only_low:.1f}%')
    coupang_n = group_stats.loc['쿠팡만', '리뷰수']
    print(f'[인사이트] 쿠팡 불만만 있는 리뷰 {coupang_n:.0f}개는 제품과 무관한 저점 → 별점 왜곡 가능성')

## 4. 브랜드/기종별 불만 유형 분포

In [ ]:
# 제품별 불만 유형 비율
product_complaint = df.groupby('product_name')[['is_coupang', 'is_product', 'is_brand_model']].mean() * 100
product_complaint.columns = ['쿠팡/배송', '제품', '브랜드/기종']
product_complaint = product_complaint.sort_values('제품', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
product_complaint.plot(kind='bar', ax=ax, colormap='Set2', alpha=0.85)
ax.set_title('제품별 불만 유형 언급 비율 (%)')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=30)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(product_complaint.round(1).to_string())

In [ ]:
# 브랜드별 비교
brand_complaint = df.groupby('brand')[['is_coupang', 'is_product', 'is_brand_model']].mean() * 100
brand_complaint.columns = ['쿠팡/배송', '제품', '브랜드/기종']

fig, ax = plt.subplots(figsize=(7, 4))
brand_complaint.plot(kind='bar', ax=ax, color=['#e53935', '#1e88e5', '#43a047'], alpha=0.85)
ax.set_title('브랜드별 불만 유형 언급 비율 (%)')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=0)
ax.legend()
plt.tight_layout()
plt.show()

print(brand_complaint.round(1).to_string())

## 5. 쿠팡 불만 샘플 vs 제품 불만 샘플

실제 텍스트를 확인해 분류가 맞는지 검증

In [ ]:
import textwrap

def show_samples(mask, label, n=3):
    subset = df[mask & (df['rating'] <= 2)].sample(min(n, mask.sum()), random_state=42)
    print(f'=== {label} 저점(1-2점) 샘플 {len(subset)}개 ===')
    for _, row in subset.iterrows():
        print(f'[{row["product_name"]} | ★{row["rating"]}] {row["title"]}')
        wrapped = textwrap.fill(str(row['content'])[:300], width=80)
        print(wrapped)
        print()

# 쿠팡만 불만
show_samples(df['is_coupang'] & ~df['is_product'], '쿠팡/배송만', n=3)

# 제품만 불만  
show_samples(~df['is_coupang'] & df['is_product'], '제품만', n=3)

## 6. Survey Answers와 불만 유형 교차 분석

구조화된 survey 응답과 텍스트 불만 유형의 일치도 확인

In [ ]:
# survey answers 펼치기
survey_rows = []
for _, row in df.iterrows():
    for ans in (row.get('reviewSurveyAnswers') or []):
        if isinstance(ans, dict) and ans.get('question'):
            survey_rows.append({
                'product_name': row['product_name'],
                'brand': row['brand'],
                'rating': row['rating'],
                'is_coupang': row['is_coupang'],
                'is_product': row['is_product'],
                'question': ans['question'],
                'answer': ans['answer'],
            })
sdf = pd.DataFrame(survey_rows)

# 가성비 응답 - 제품 불만 유형별
q = '가성비'
sub = sdf[sdf['question'] == q]
cross = pd.crosstab(sub['is_product'], sub['answer'])
cross_pct = cross.div(cross.sum(axis=1), axis=0) * 100
cross_pct.index = ['제품불만 없음', '제품불만 있음']
print(f'=== {q} 응답 × 제품불만 유형 ===')
print(cross_pct.round(1).to_string())

fig, ax = plt.subplots(figsize=(9, 3))
cross_pct.plot(kind='bar', ax=ax, alpha=0.85)
ax.set_title(f'가성비 응답 — 제품불만 언급 여부별 (%)')
ax.set_ylabel('%')
ax.tick_params(axis='x', rotation=0)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 7. 인사이트 요약

In [ ]:
# 수치 기반 인사이트 자동 생성
total = len(df)
low = df[df['rating'] <= 2]

# 쿠팡만 불만인 저점 리뷰
coupang_only_low = df[(df['is_coupang']) & (~df['is_product']) & (df['rating'] <= 2)]

# 제품 불만이지만 고점인 리뷰 (긍/부 혼재)
product_but_high = df[(df['is_product']) & (df['rating'] >= 4)]

print('=== 인사이트 요약 ===')
print()
print(f'① 별점 왜곡 규모')
print(f'   저점(1-2점) 리뷰 {len(low):,}개 중 "쿠팡/배송만" 불만: {len(coupang_only_low):,}개 ({len(coupang_only_low)/len(low)*100:.1f}%)')
print(f'   → 이 리뷰들은 제품 품질과 무관하게 별점이 낮아진 케이스')
print()
print(f'② 제품 불만이면서도 고점인 리뷰')
print(f'   제품 불만 키워드가 있지만 4-5점: {len(product_but_high):,}개 ({len(product_but_high)/len(df[df["is_product"]])*100:.1f}%)')
print(f'   → "아쉽지만 전반적으로 만족" 유형, 긍/부 혼재')
print()
print(f'③ 브랜드별 불만 패턴 차이')
for brand in ['Apple', 'Samsung']:
    sub = df[df['brand'] == brand]
    c_pct = sub['is_coupang'].mean() * 100
    p_pct = sub['is_product'].mean() * 100
    print(f'   {brand}: 쿠팡불만 {c_pct:.1f}%, 제품불만 {p_pct:.1f}%')